In [34]:
# INF201 - Deliverable 1
# Task 4: HeatWaves

In [ ]:
import requests
import pandas as pd
import yaml

"""This function creates an yml file that tells you the dates, station id, station name and year
   of when different heatwaves happens throughout that place and year. A 5 day-period of 27 degress celsius 
   and over is classified as a heatwave. It requests data from frost.met.no and uses boolean values to check
   hot days and cold days to find heatwaves and the dates to those days. Then it creates the yml file.
   heatwaves(station_id, year) is where you can choose which station and year to want to search for."""

def heatwaves(station_id,year):
    with open('secrets.txt', 'r') as f:
       client_id = f.read().strip() 
 
    #Request a stations daily max temperature for the given year and the stations metdata from frost.met.no, 
    #and pritns error if request fails
    endpoint = 'https://frost.met.no/observations/v0.jsonld'
    endpoint2 = 'https://frost.met.no/sources/v0.jsonld'
    parameters = {
    'sources': f"{station_id}",
    'elements': 'max(air_temperature P1D)',
    'referencetime': f'{year}-01-01/{year}-12-31',
    }
    parameters2 = { 'ids': station_id, }

    r = requests.get(endpoint, parameters, auth=(client_id,''))
    r1 = requests.get(endpoint2, parameters2, auth=(client_id,''))

    json = r.json()
    json1 = r1.json()

    if r.status_code == 200:
        data = json['data']
    else:
        print('Error! Returned status code %s' % r.status_code)
        print('Message: %s' % json['error']['message'])
        print('Reason: %s' % json['error']['reason'])
        return

    if r1.status_code == 200:
        data1 = json1['data']
    else:
        print('Error! Returned status code %s' % r1.status_code)
        print('Message: %s' % json1['error']['message'])
        print('Reason: %s' % json1['error']['reason'])
        return

    #Goes through obersvations and stations records to build two lists of dicts, 
    #then turns them into dataframe df and df1, also dfs "time" to datatime(for dates).
    rows_data = [] #empty list
    rows_data1 = []
    for item in data:
        row = {  #dict
            "time": item["referenceTime"], #Here i am just saying that i named "time" out of "referenceTime", because that is where the "date" is located
            "temperature": item["observations"][0]["value"], #Same logic for temperature, but that is in "observations", and must be the first [0] which representes 'elementId': 'max(air_temperature P1D)', and [value] gives me the value for the max temperature that day, which comes right after [0]
            "exact_station_id" : item["sourceId"],
    }
        rows_data.append(row) #i just append the dict i made in the list "rows"

    for item in data1:
        row1 = {"city_name" : item['name'],
        }
        rows_data1.append(row1)

    df = pd.DataFrame(rows_data) #here header shows the first 5 rows
    df1 = pd.DataFrame(rows_data1)
    df["time"]= pd.to_datetime(df["time"]) #This converts the arguments in time into actual dates

    #Used boolean values to track hot and cold days(27°C and over is hot), tracks heatwaves and dates to them
    hw = df["temperature"] >= 27 #Says that values over and alike 27 degress are True, and under is False.
    streak = 0
    count_heatwaves = 0
    heatwave_dates = [] 
    for i, is_hot in enumerate(hw): #is_hot is as a variable taker in hw.
        if is_hot == 1: #if is_hot is set as the value 1, cus thats what python has for the value of "True"
            streak += 1 #gives streak one value
        if is_hot == 0 and streak >= 5: #Gives us the value that ended a heatwave, (under 27 degress)
            end_date = df["time"][i-1] #Gives the date when the heatwave ends, thats why its i-1, 
            heatwave_dates.append(f"{start_date.date()} - {end_date.date()}") #Puts it in a list, statdate and endate
        if is_hot == 0: #Value for False is 0 in python, so when temperature is under 27 then is_hot = 0, and it restarts the streak.
            streak = 0
        if streak == 5: #This if-statement says that if streak reaches 5 days in a row, it gets added to the count_heatwaves as one value.
            count_heatwaves += 1
        if is_hot == 1 and streak == 5: #Gives us the 5 value-day in a heatwave, when its classified as a "heatwave"
            start_date = df["time"][i-4] #Gives us then a start date, because is a heatwave starts at day 5, "i-4" goes back to the first date.

    if streak >= 5:
        heatwave_dates.append(f"{start_date.date()} - {df["time"][i].date()}") #This is for if there's a heatwave at the end of the year, which dosent get followed up by a cold day.


    #Looks up stationID, and prints which station and year, and total heatwaves in that year and place.
    exact_station = df["exact_station_id"][0].split(":0")
    city = df1["city_name"][0]
    print(f"Total heatwaves in {station_id} {year}: {count_heatwaves}")
    
    #Creates a summary dict involving dates,stationID,station Name, and year, sends it to a yaml file which is
    #names after that station ID and year
    summary = {
        "dates" : heatwave_dates,
        "station_id": exact_station[0],
        "station_name": city,
        "year" : year,
        
    }
    
    with open (f"{station_id}_{year}.yml", "w") as file:
        yaml.dump(summary, file)
    return

heatwaves("SN17850", 2025) #Put whatever station-id and year to get the heatwaves in that place and year.


Total heatwaves in SN17850 2025: 1
